# Calibration (host noise-floor): CS-01 TAS

**Purpose**: characterise the per-host noise floor that every `experiment` run depends on. Four probes (timer resolution, scheduling jitter, loopback latency, empty-handler scaling) produce one JSON envelope that `05-experimental.ipynb` and `06-yoly-experimental.ipynb` reference when reporting measured latencies.

**Inputs**: none. Host identity (hostname, OS, CPU count, RAM) is snapshotted at runtime.

**Outputs**:
- `data/results/experiment/calibration/<host>_<YYYYMMDD_HHMMSS>.json`: the full calibration envelope.
- `data/img/experiment/calibration/dashboard.{png,svg}`: 2x2 summary card (timer / jitter / loopback / handler scaling).
- `data/img/experiment/calibration/scaling.{png,svg}`: standalone empty-handler scaling line plot.

**Equivalent CLI**:
```bash
python src/methods/calibration.py
```

**How to use the baseline**: every measured experiment latency on this host should be reported as:

```
reported = measured_us - loopback.median_us  +/- jitter.p99_us
```

i.e. subtract the loopback median (host overhead), report the jitter p99 as the uncertainty band. Any measured value below `loopback.median_us` is an instrument error, not a real service. Full plan + checkpoint log in [notes/calibration.md](notes/calibration.md).

This notebook is thin: all logic lives in `src.methods.calibration` (runner) and `src.view.characterization` (plotters). The cells below just call `run()` and display.

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd

from src.io import (
    load_dim_card,
    load_latest_calibration,
    load_method_cfg,
)
from src.methods.calibration import derive_calib_coefs, derive_calib_sweep, run
from src.view import (
    plot_calib_dashboard,
    plot_calib_rate_sweep,
    plot_calib_scaling,
    plot_arts_distributions,
    plot_yoly_arts_behaviour,
    plot_yoly_arts_charts,
    plot_yoly_chart,
)

_IMG_ROOT = Path("data/img/experiment/calibration")

# Toggle the rate-saturation probe. Default False so a rerun stays under ~3 min;
# flip to True to also run the experiment-method at the configured rate ladder
# x trials_per_rate (adds ~10-15 min wall time depending on the ladder).
_RUN_RATE_SWEEP = True

# Per-request body size (bytes) for the phi coefficient in the dim card.
# Sourced from data/config/method/calibration.json::payload_size_bytes
# (e.g. 131072 = 128 kB). 0 keeps phi NaN (degenerate-memory case).
_PAYLOAD_BYTES = int(load_method_cfg("calibration").get("payload_size_bytes", 0))


## 1. Run the four probes

`run()` executes under `winmm.timeBeginPeriod(1)` on Windows (no-op elsewhere). Default sample counts take ~60-90 s on a 16-core laptop; tune via keyword args if needed (e.g. `jitter_samples=500, loopback_samples=500, per_worker=20, n_con_usr=(1, 10)` for a ~8 s smoke).

The envelope is written to `data/results/experiment/calibration/<host>_<timestamp>.json` and returned in-memory for the display cells below.

In [ ]:
_envelope = run(write=True, verbose=True,
                skip_rate_sweep=not _RUN_RATE_SWEEP)
print(f"\nwrote: {_envelope.get('output_path')}")


## 2. Host profile

The machine the rest of the calibration was measured against. Record thermals / background load conditions in [notes/devlog.md](notes/devlog.md) when benching formally.

In [ ]:
_host = _envelope["host_profile"]
pd.Series({
    "hostname": _host.get("hostname"),
    "os": _host.get("os"),
    "python": f"{_host.get('python')} ({_host.get('python_impl')})",
    "cpu_count": _host.get("cpu_count"),
    "cpu_machine": _host.get("cpu_machine"),
    "ram_total_gb": round(float(_host.get("ram_total_gb") or 0), 2),
    "timestamp": _envelope.get("timestamp"),
    "elapsed_s": _envelope.get("elapsed_s"),
}, name="value")

## 3. Probe summary tables

Each probe reports a distribution summary. Acronyms expanded below:

- **min**: minimum observed value; for `timer` this is the actual clock resolution.
- **median / p50**: 50th percentile (half the samples are faster, half slower).
- **mean**: arithmetic mean.
- **p95 / p99**: 95th / 99th percentile (worst-case tail behaviour).
- **max**: single worst sample.
- **std**: standard deviation (dispersion).
- **ns**: nanoseconds (10^-9 s).  **us**: microseconds (10^-6 s).

In [ ]:
_timer = _envelope.get("timer", {})
_jitter = _envelope.get("jitter", {})
_loopback = _envelope.get("loopback", {})

pd.DataFrame({
    "timer [ns]": {
        "min": _timer.get("min_ns"),
        "median / p50": _timer.get("median_ns"),
        "mean": _timer.get("mean_ns"),
        "std": _timer.get("std_ns"),
    },
    "jitter [us]": {
        "mean": _jitter.get("mean_us"),
        "median / p50": _jitter.get("p50_us"),
        "p99": _jitter.get("p99_us"),
        "max": _jitter.get("max_us"),
    },
    "loopback [us]": {
        "min": _loopback.get("min_us"),
        "median / p50": _loopback.get("median_us"),
        "p95": _loopback.get("p95_us"),
        "p99": _loopback.get("p99_us"),
    },
}).round(2)

## 4. Handler scaling table

Empty `/ping` endpoint, keep-alive, measured at increasing in-flight request counts. Growing gap between `c=1` and `c=10` indicates how much the FastAPI event loop queues when concurrent requests stack up; often the real driver of prototype latency rise at high rates.

In [ ]:
_handler = _envelope.get("handler_scaling", {})
_rows = []
for _c, _stats in sorted(_handler.items(), key=lambda _p: int(_p[0])):
    _rows.append({
        "n_con_usr": int(_c),
        "min [us]": _stats.get("min_us"),
        "median / p50 [us]": _stats.get("median_us"),
        "p95 [us]": _stats.get("p95_us"),
        "p99 [us]": _stats.get("p99_us"),
        "samples": _stats.get("samples"),
    })
pd.DataFrame(_rows).set_index("n_con_usr").round(1)

## 5. Calibration dashboard (single-figure summary)

2x2 card combining the three headline bar charts with the handler-scaling line plot. The suptitle carries the host identity, timestamp, and the `reported = measured - loopback_median +/- jitter_p99` formula so the figure is usable on its own as a dissertation appendix.

In [ ]:
plot_calib_dashboard(
    _envelope,
    file_path=str(_IMG_ROOT),
    fname="dashboard.png",
)

## 6. Handler-scaling detail

Standalone line plot: median (p50), 95th percentile (p95), 99th percentile (p99) vs `n_con_usr` (concurrent users, in-flight requests) on the CLIENT side. The SERVICE stays at `c_srv=1` (single uvicorn worker) throughout. Log-log axes so the span from `n_con_usr=1` to `n_con_usr=10000` is legible.

In [ ]:
plot_calib_scaling(
    _envelope.get("handler_scaling", {}),
    file_path=str(_IMG_ROOT),
    fname="scaling.png",
)

## 6b. Dimensional coefficient card (Route B: measured)

Host-specific dimensionless signature of the single-worker (`c_srv=1`) calibration `/ping` service. Coefficients are derived directly from the measured `handler_scaling` and `loopback` blocks (Route B — measurement, not M/M/c/K prediction) using `src.methods.calibration.derive_calib_coefs`. `mu` comes from the idle loopback probe (`mu = 1e6 / loopback.median_us`); the handler-scaling curve supplies `R`, `X`, `L`, `Wq` at each `n_con_usr` level.

- **theta = L / K**: queue occupancy.
- **sigma = W * lambda / K**: Stall; queueing share of capacity.
- **eta = X * K / (mu * c_srv)**: effective-yield vs host service-rate ceiling.
- **phi = (L * payload) / (K * payload) = L / K**: memory-usage; degenerate here because every `/ping` request carries the same payload (so `phi == theta`). Populated with `NaN` when `payload_size_bytes=0`; becomes informative only with the payload-echo upgrade.
- **epsilon** is structurally 0 (the service has no business logic that can fail) and is therefore NOT part of this card.

The 2x2 figure rendered below reuses `src.view.dc_charts.plot_yoly_chart` — the same plotter the TAS dimensional method uses on actual architecture coefficients. Saved as `dim_card.{png,svg}`.

In [ ]:
_dim_card = _envelope.get("dimensional_card")
if not _dim_card:
    _dim_card = derive_calib_coefs(_envelope, payload_size_bytes=_PAYLOAD_BYTES)

if not _dim_card:
    print("dimensional_card unavailable -- loopback or handler_scaling missing.")
else:
    _meta = _dim_card.get("meta", {})
    print(f"pipeline       : {_meta.get('pipeline')}")
    print(f"payload_size_B : {_meta.get('payload_size_bytes')}")
    print(f"mu_req_per_s   : {_meta.get('mu_req_per_s'):.2f}")
    print(f"n_con_usr      : {_meta.get('n_con_usr')}")

    _logscale = bool(_PAYLOAD_BYTES)  # phi is NaN when payload=0 -> linear scale
    plot_yoly_chart(
        _dim_card,
        paths={"Calibration": "CALIB"},
        logscale=_logscale,
        title="Calibration dimensional card (Route B, PyDASA-routed)",
        file_path=str(_IMG_ROOT),
        fname="dim_card.png",
        verbose=False,
    )


## 6c. Dimensional sweep (Route A: predicted)

Companion to 6b. Cell 6b shows what THIS host actually does at one
operating point (Route B, measured). This cell shows what M/M/c/K
predicts the same calibration service WOULD do across a Cartesian
design grid `(mu_factor x c x K)`, ramping $\lambda$ per combo. Grid
lives in `data/config/method/calibration.json::sweep_grid`. The gap
between the two charts is the Route-A vs Route-B agreement gap; if it
is large, real host effects (GIL, scheduler, accept buffer) dominate
over the Markovian assumptions.


In [ ]:
_sweep = derive_calib_sweep(_envelope)
if not _sweep:
    print("sweep unavailable -- loopback block missing or sweep_grid empty.")
else:
    print(f"combos: {len(_sweep)} (one cloud per (c, K) pair; mu_factor + lambda ramp inside each)")
    # flatten nested {combo_tag: {symbol: array}} into one dict + scenarios map (one display label per combo)
    _overlay = {}
    _scenarios = {}
    for _combo_tag, _block in _sweep.items():
        _overlay.update(_block)
        # display label like 'c=1, K=50' parsed from CALIB_c1_K50
        _label = _combo_tag.replace("CALIB_", "").replace("c", "c=").replace("_K", ", K=")
        _scenarios[_label] = _combo_tag
    plot_yoly_chart(
        _overlay,
        scenarios=_scenarios,
        title=r"Calibration sweep (Route A): predicted M/M/c/K design space across $(\mu, c, K, \lambda)$",
        file_path=str(_IMG_ROOT),
        fname="dim_card_sweep.png",
    )


## 7. Rate saturation (optional)

Only populated when `_RUN_RATE_SWEEP = True` in the setup cell. Drives the full experiment mesh at the configured rate ladder (default `[100, 200, 300, 400, 500]` req/s x 5 trials) and reports the highest rate the mesh sustains under the configured loss target. Figure saved as `rate_sweep.{png,svg}` alongside the dashboard.


In [ ]:
_rate_sweep = _envelope.get("rate_sweep")
if _rate_sweep is None:
    print("rate_sweep block absent -- set _RUN_RATE_SWEEP = True and re-run "
          "the probe cell to populate it.")
else:
    _rows = []
    for _k in sorted(_rate_sweep["aggregates"].keys(), key=lambda _x: float(_x)):
        _a = _rate_sweep["aggregates"][_k]
        _rows.append({
            "target [req/s]": float(_k),
            "effective mean": _a.get("mean"),
            "effective range": f'[{_a.get("lo", 0):.2f}, {_a.get("hi", 0):.2f}]',
            "mean loss [%]": _a.get("mean_loss_pct"),
            "entry lambda mean": _a.get("mean_entry_lambda"),
            "trials": _a.get("n"),
        })
    display(pd.DataFrame(_rows).set_index("target [req/s]").round(3))
    _cal = _rate_sweep.get("calibrated_rate")
    _target = _rate_sweep.get("target_loss_pct")
    if _cal is None:
        print(f"\nNo rate cleared the {_target}% mean-loss bar.")
    else:
        print(f"\nHighest sustainable rate at <= {_target}% mean loss: "
              f"{float(_cal):.1f} req/s")

    plot_calib_rate_sweep(
        _rate_sweep,
        file_path=str(_IMG_ROOT),
        fname="rate_sweep.png",
    )


## 8. Apply the baseline to experiment results

The two numbers that matter when reporting later experiments on this host:


In [ ]:
_floor = float(_loopback.get("median_us", 0.0))
_band = float(_jitter.get("p99_us", 0.0))
print(f"Loopback median (host overhead) : {_floor:,.1f} us")
print(f"Jitter p99 (uncertainty band)   : {_band:,.1f} us")
print()
print("Example: a measured experiment latency of 8300 us reports as:")
print(f"  {8300 - _floor:,.1f} us  +/- {_band:,.1f} us")
print()
print("Any measured latency below the host overhead is an instrument error.")

## Summary

Run this notebook before every fresh `05-experimental.ipynb` / `06-yoly-experimental.ipynb` session so the downstream cells can load the latest calibration JSON by timestamp. A stale baseline is fine for iteration; a dissertation-quality run pairs every experiment result with a same-session calibration.

**Next in the pipeline**: `05-experimental.ipynb` (fixed-point per-adaptation execution) and `06-yoly-experimental.ipynb` (configuration-sweep yoly).